In [34]:
from langgraph.graph import StateGraph, START, END  
# StateGraph : 그래프는 상태를 가진다는 것 (state 상태란 그래프를 통해 이동하는 데이터임)
# START, END는 예약어 같은거임 
from typing_extensions import TypedDict 

class State(TypedDict):
    hello : str 
    a: bool

graph_builder=StateGraph(State)


In [ ]:
# 노드는 그 자리에서 state를 받을 수 있음. (당연한말)
# 아래 노드 3개 생성(이게 우리 작업 단위임)
def node_one(state : State): 
    print("node_one", state)
    return{ # state 수정 하기
        "hello" : "from node one.",
        "a" : True # a를 굳이 다음으로 보내지 않더라도 알아서 node_two로 넘어감
    }

def node_two(state : State):
    print("node_two", state)
    return {"hello":"from node two"}

def node_three(state : State):
    print("node_three", state)
    return {"hello":"from node three"}


In [36]:
# 우리는 위에 만든 node를 그래프에게 줄것임(graph_builder)

graph_builder.add_node("node_one", node_one)  # node_one이라는 이름으로 node_one함수 실행  
graph_builder.add_node("node_two", node_two) 
graph_builder.add_node("node_three", node_three) 

# node들 끼리 연결하기 위해 edge를 만든다(edge는 화살표) 
graph_builder.add_edge(START, "node_one") # start에서 "node_one"으로 화살표 추가한다는 뜻
graph_builder.add_edge("node_one", "node_two")
graph_builder.add_edge("node_two", "node_three")
graph_builder.add_edge("node_three", END)

# 이렇게 하면 그래프 완성됨 


In [37]:
# 완성된 그래프 compile하기 (컴파일을 하면 langgraph는 해당 graph가 유효한지 검사할 수있다.)
# 내 그래프의 edge들이 말이 되는지 검사 

graph = graph_builder.compile()

result = graph.invoke(
    {"hello":"world", #state에 뭐 넣기 
     "hellobaby":"worldbaby"}  # 위에 정해둔 state가 아니면 print해도 출력이 안됨 
)  

node_one {'hello': 'world'}
node_two {'hello': 'from node one.', 'a': True}
node_three {'hello': 'from node two', 'a': True}


In [38]:
result

{'hello': 'from node three', 'a': True}